In [33]:
import scanpy as sc
import anndata as ad
import magic
import torch
import numpy as np
import pandas as pd
import scvelo as scv # If you are doing RNA velocity

from magic import MAGIC
import os

os.chdir("/Users/dm954/Documents/code/mixed_diffusion/benchmarks/neo_cortex")
print(os.getcwd())

/Users/dm954/Documents/code/mixed_diffusion/benchmarks/neo_cortex


In [34]:
adata_test = ad.read_h5ad("/Users/dm954/Documents/code/mixed_diffusion/data/neo_cortex/polioudakis_cleaned.h5ad")
adata_train = ad.read_h5ad("/Users/dm954/Documents/code/mixed_diffusion/data/neo_cortex/nowakowski_cleaned.h5ad")

print(adata_test.shape)
print(adata_test)
PCA_DIM = 15

(15126, 785)
AnnData object with n_obs × n_vars = 15126 × 785
    obs: 'cell_type_transferred'


In [35]:
from sklearn.decomposition import PCA

# Run MAGIC denoising and save denoised embeddings as .pt

# Helper to get dense numpy arrays
def to_array(X):
    try:
        if hasattr(X, 'toarray'):
            return X.toarray()
        else:
            return np.asarray(X)
    except Exception:
        return np.asarray(X)

# Fetch raw matrices
X_test_raw = to_array(adata_test.X)

# Run MAGIC denoising
magic_op = MAGIC(n_pca=PCA_DIM)
print('Running MAGIC on train...')
X_test_magic = MAGIC(n_pca=PCA_DIM).fit_transform(X_test_raw)

# Build categorical codes and label encoder (number -> name)
cat_labels = pd.Categorical(adata_test.obs['cell_type_transferred'])
label_test = cat_labels.codes  # integer codes per cell
label_encoder = {str(cat): int(i) for i, cat in enumerate(cat_labels.categories)}

# Apply PCA to reduce X_test_magic to PCA_DIM dimensions
pca = PCA(n_components=PCA_DIM)
X_test_magic = pca.fit_transform(X_test_magic)

X_test_raw = pca.fit_transform(X_test_raw)

print(X_test_magic.shape, X_test_raw.shape)

test_magic = {
    'x_denoised': torch.tensor(X_test_magic, dtype=torch.float32),
    'x_true': torch.tensor(X_test_raw, dtype=torch.float32),
    'x_denoised_labels': torch.tensor(label_test),
    'data_config': { 'label_encoder': label_encoder }
}

out_test = 'magic/denoising_results.pt'
torch.save(test_magic, out_test)
print(f"Saved MAGIC denoised file: '{out_test}'")

Running MAGIC on train...
Calculating MAGIC...
  Running MAGIC on 15126 cells and 785 genes.
  Calculating graph and diffusion operator...
    Calculating PCA...
    Calculated PCA in 0.67 seconds.
    Calculating KNN search...
    Calculated KNN search in 0.82 seconds.
    Calculating affinities...
    Calculated affinities in 0.84 seconds.
  Calculated graph and diffusion operator in 2.34 seconds.
  Calculating imputation...
  Calculated imputation in 0.30 seconds.
Calculated MAGIC in 2.65 seconds.
(15126, 15) (15126, 15)
Saved MAGIC denoised file: 'magic/denoising_results.pt'


In [36]:
from pyALRA import alra, normalize_data, choose_k

# Load your data into an AnnData object

# count_matrix = adata_test.layers['counts'].toarray()
normalized_count_matrix = adata_test.X

# Determine the optimal k
k = choose_k(normalized_count_matrix)

# Apply ALRA
embeddings = alra(normalized_count_matrix, k['k'])['A_norm_rank_k_cor_sc']
print(embeddings.shape)
print(k)

# Apply PCA to reduce embeddings to PCA_DIM dimensions
pca = PCA(n_components=PCA_DIM)
X_alra_denoised = pca.fit_transform(embeddings)
X_alra_raw = pca.fit_transform(to_array(adata_test.X))

print(X_alra_denoised.shape, X_alra_raw.shape)

# Build categorical codes and label encoder
cat_labels_alra = pd.Categorical(adata_test.obs['cell_type_transferred'])
label_alra = cat_labels_alra.codes

alra_results = {
    'x_denoised': torch.tensor(X_alra_denoised, dtype=torch.float32),
    'x_true': torch.tensor(X_alra_raw, dtype=torch.float32),
    'x_denoised_labels': torch.tensor(label_alra),
    'data_config': {'label_encoder': label_encoder}
}

out_alra = 'alra/denoising_results.pt'
torch.save(alra_results, out_alra)
print(f"Saved ALRA denoised file: '{out_alra}'")


Read matrix with 15126 cells and 785 genes
Find the 0.001 quantile of each gene
Sweep
Scaling all except for 1 columns
0.00% of the values became negative in the scaling process and were set to zero
The matrix went from 17.44% nonzero to 48.39% nonzero
(15126, 785)
{'k': np.int64(19), 'num_of_sds': array([ 1.34854092e+04,  1.51984656e+03,  4.21887543e+02,  8.53670044e+01,
        2.04021652e+02,  2.26258835e+02,  1.93637436e+02,  6.73379135e+01,
        5.34404297e+01,  1.14374969e+02,  7.01282043e+01,  1.02253742e+01,
        4.10387383e+01,  3.21120186e+01,  2.65870247e+01,  1.88523865e+01,
        8.03388023e+00,  1.00466935e-02,  9.72047329e+00,  5.59923202e-02,
        9.36721146e-01,  1.88104773e+00,  8.37944329e-01,  1.14078486e+00,
        2.23734546e+00,  7.55442500e-01, -9.68707800e-01,  2.29055214e+00,
       -5.43022230e-02, -3.47628117e-01,  1.77914107e+00, -4.56295162e-01,
       -5.47685623e-01,  1.15468121e+00, -5.22897542e-01,  1.61413741e+00,
        2.31136546e-01, -

In [37]:
# Run sklearn NMF on test data and save results (using non-negative matrix factorization)
import os
import numpy as np
from sklearn.decomposition import NMF, PCA

print(f"Running sklearn NMF on adata_test with n_components={PCA_DIM}...")

# Use counts layer if available, otherwise adata_test.X
X_counts = to_array(adata_test.layers['counts']) if 'counts' in adata_test.layers else to_array(adata_test.X)
# Ensure non-negative
X_counts = np.clip(X_counts, a_min=0.0, a_max=None)
# Apply log1p transform (preserve non-negativity)
X_counts_log = np.log1p(X_counts)

# Fit NMF on log-transformed counts
nmf_model = NMF(n_components=PCA_DIM, init='nndsvda', random_state=0)
W = nmf_model.fit_transform(X_counts_log)  # cells x components
H = nmf_model.components_

# Prepare x_true: apply log1p to raw and reduce with PCA for comparability
X_raw = to_array(adata_test.X)
X_raw_log = np.log1p(X_raw)
pca_nmf = PCA(n_components=PCA_DIM)


# Build categorical codes and label encoder
cat_labels_nmf = pd.Categorical(adata_test.obs['cell_type_transferred'])
label_nmf = cat_labels_nmf.codes
if 'label_encoder' not in globals():
    label_encoder = {str(cat): int(i) for i, cat in enumerate(cat_labels_nmf.categories)}

nmf_results = {
    'x_denoised': torch.tensor(W, dtype=torch.float32),
    'x_true': torch.tensor(W, dtype=torch.float32),
    'x_denoised_labels': torch.tensor(label_nmf),
    'data_config': {'label_encoder': label_encoder}
}

out_nmf = 'nmf/denoising_results.pt'
# ensure output dir
out_dir = os.path.dirname(out_nmf)
if out_dir and not os.path.exists(out_dir):
    os.makedirs(out_dir, exist_ok=True)

torch.save(nmf_results, out_nmf)
print(f"Saved sklearn NMF denoised file: '{out_nmf}'")

Running sklearn NMF on adata_test with n_components=15...
Saved sklearn NMF denoised file: 'nmf/denoising_results.pt'


In [38]:
from scipy.sparse import issparse
def knn_moments(adata, n_neighbors=30, n_pcs=50, layer=None):
    """
    Smooth expression data using kNN neighbors' mean and variance.
    
    Parameters
    ----------
    adata : AnnData
        Annotated data matrix
    n_neighbors : int
        Number of neighbors to use for smoothing
    n_pcs : int
        Number of PCs to use for kNN computation
    layer : str or None
        Layer to smooth. If None, uses adata.X
        
    Returns
    -------
    Modifies adata in place:
    - adata.layers["Ms"] : smoothed mean expression (kNN-averaged)
    - adata.layers["Vs"] : smoothed variance (kNN-averaged)
    """
    
    # Compute neighbors if not already computed
    print("Computing neighbors...")
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=n_pcs)
    
    # Get the data to smooth
    if layer is None:
        X = adata.X
    else:
        X = adata.layers[layer]
    
    # Convert to dense if sparse
    if issparse(X):
        X = X.toarray()
    
    # Get neighbor indices from the distances/connectivities
    connectivities = adata.obsp["connectivities"]
    
    n_obs, n_vars = X.shape
    
    # Initialize smoothed mean and variance
    Ms = np.zeros_like(X, dtype=np.float32)
    Vs = np.zeros_like(X, dtype=np.float32)
    
    print("Computing kNN moments...")
    
    # For each cell, compute mean and variance over its neighbors
    for i in range(n_obs):
        # Get neighbor indices (including self)
        neighbors = connectivities[i].nonzero()[1]
        
        if len(neighbors) > 0:
            # Get neighbor expression values
            neighbor_expr = X[neighbors, :]
            
            # Compute mean over neighbors
            Ms[i, :] = np.mean(neighbor_expr, axis=0)
            
            # Compute variance over neighbors
            Vs[i, :] = np.var(neighbor_expr, axis=0)
        
        if (i + 1) % 1000 == 0:
            print(f"  Processed {i + 1}/{n_obs} cells")
    
    # Store results
    adata.layers["Ms"] = Ms
    adata.layers["Vs"] = Vs
    
    print("Done!")
    return adata

In [40]:
import os
import numpy as np
from sklearn.decomposition import NMF, PCA

print(f"Running adata_test with n_components={PCA_DIM}...")

for k in [3, 5, 7, 10, 15, 30]:

    knn_moments(adata_test, n_neighbors=k, n_pcs=PCA_DIM)
    print(adata_test.layers)

    W = adata_test.layers["Ms"].copy() 


    knn_results = {
        'x_denoised': torch.tensor(W, dtype=torch.float32),
        'x_true': torch.tensor(W, dtype=torch.float32),
        'x_denoised_labels': torch.tensor(label_nmf),
        'data_config': {'label_encoder': label_encoder}
    }

    out_knn = f"knn_{k}/denoising_results.pt"
    # ensure output dir
    out_dir = os.path.dirname(out_knn)
    if out_dir and not os.path.exists(out_dir):
        os.makedirs(out_dir, exist_ok=True)

    torch.save(knn_results, out_knn)
    print(f"Saved sklearn NMF denoised file: '{out_knn}'")

Running adata_test with n_components=15...
Computing neighbors...
Computing kNN moments...
  Processed 1000/15126 cells
  Processed 2000/15126 cells
  Processed 3000/15126 cells
  Processed 4000/15126 cells
  Processed 5000/15126 cells
  Processed 6000/15126 cells
  Processed 7000/15126 cells
  Processed 8000/15126 cells
  Processed 9000/15126 cells
  Processed 10000/15126 cells
  Processed 11000/15126 cells
  Processed 12000/15126 cells
  Processed 13000/15126 cells
  Processed 14000/15126 cells
  Processed 15000/15126 cells
Done!
Layers with keys: Ms, Vs
Saved sklearn NMF denoised file: 'knn_3/denoising_results.pt'
Computing neighbors...
Computing kNN moments...
  Processed 1000/15126 cells
  Processed 2000/15126 cells
  Processed 3000/15126 cells
  Processed 4000/15126 cells
  Processed 5000/15126 cells
  Processed 6000/15126 cells
  Processed 7000/15126 cells
  Processed 8000/15126 cells
  Processed 9000/15126 cells
  Processed 10000/15126 cells
  Processed 11000/15126 cells
  Proc

In [ ]:
import os
import numpy as np
from sklearn.decomposition import NMF, PCA

print(f"Running adata_test with just PCA")


# Apply PCA to reduce X_test_magic to PCA_DIM dimensions
pca = PCA(n_components=PCA_DIM)
W = pca.fit_transform(adata_test.X)
pca_results = {
    'x_denoised': torch.tensor(W, dtype=torch.float32),
    'x_true': torch.tensor(W, dtype=torch.float32),
    'x_denoised_labels': torch.tensor(label_nmf),
    'data_config': {'label_encoder': label_encoder}
}

out_pca = f"pca/denoising_results.pt"
# ensure output dir
out_dir = os.path.dirname(out_pca)
if out_dir and not os.path.exists(out_dir):
    os.makedirs(out_dir, exist_ok=True)

torch.save(pca_results, out_pca)
print(f"Saved denoised file: '{out_pca}'")

Running adata_test with just PCA
Saved denoised file: 'pca/denoising_results.pt'
